# SERPENS Black Hole Test
Test notebook for GR-enabled simulation using the `StellarBH-10` system.

In [1]:
import os
WORKDIR = '/Users/raghavchari/SERPENS'
if os.getcwd() != WORKDIR:
    os.chdir(WORKDIR)
print('Working Directory:', os.getcwd())

Working Directory: /Users/raghavchari/SERPENS


In [2]:
from src.serpens_simulation import SerpensSimulation
from src.parameters import GLOBAL_PARAMETERS
from src.species import Species
import numpy as np

## 1. Enable GR and create the simulation
Set `gr_enabled=True` and point `gr_source` at the black hole before initializing.

In [3]:
# Enable GR corrections
GLOBAL_PARAMETERS.set('gr_enabled', True)
GLOBAL_PARAMETERS.set('gr_source', 'bh')

# Create simulation with 10 solar mass stellar black hole system
sim = SerpensSimulation(system='StellarBH-10')

Initializing new simulation instance...
GR enabled: r_schwarzschild = 2.95e+04 m, r_isco = 8.86e+04 m
	 	 ... done!


## 2. Inspect the system
Print out bodies, the Schwarzschild radius, and ISCO.

In [4]:
print(f'Number of bodies: {sim.N}')
print(f'N_active (gravitating): {sim.N_active}\n')

for i in range(sim.N):
    p = sim.particles[i]
    print(f'Particle {i}: m={p.m:.3e} kg, r={p.r:.3e} m, pos=({p.x:.3e}, {p.y:.3e}, {p.z:.3e})')

print(f'\nr_schwarzschild = {sim._r_schwarzschild:.3e} m')
print(f'r_isco          = {sim._r_isco:.3e} m')
print(f'r_isco / r_s    = {sim._r_isco / sim._r_schwarzschild:.1f}')

Number of bodies: 3
N_active (gravitating): 3

Particle 0: m=1.988e+31 kg, r=2.954e+04 m, pos=(-1.227e+11, 0.000e+00, 0.000e+00)
Particle 1: m=1.988e+30 kg, r=6.963e+08 m, pos=(1.227e+12, 0.000e+00, 0.000e+00)
Particle 2: m=1.000e+10 kg, r=5.000e+05 m, pos=(-1.222e+11, 0.000e+00, 0.000e+00)

r_schwarzschild = 2.954e+04 m
r_isco          = 8.863e+04 m
r_isco / r_s    = 3.0


## 3. Define a species and source
Use Hydrogen as a test particle species sputtered from the disk source.

In [ ]:
import numpy as np

# Compute disk source orbital velocity for sputter scaling
G = 6.6743e-11
v_orb = np.sqrt(G * sim.particles['bh'].m / 5.0e8)
print(f"Disk source orbital velocity: {v_orb:.2e} m/s")

# Use sputter velocities as a meaningful fraction of v_orb so particles disperse
H = Species(
    name='H',
    n_th=0,
    n_sp=80,
    mass_per_sec=1000,
    beta=0.0,
    lifetime=1e8,
    sput_spec={
        'model_smyth_v_b': 0.05 * v_orb,   # 5% of orbital velocity
        'model_smyth_v_M': 0.30 * v_orb    # up to 30%
    }
)
print(H)

In [6]:
sim.object_to_source('disk_source', H)

## 4. Run a short test simulation
Advance for a few spawning steps to verify GR integration and ISCO removal work.

In [ ]:
# Run several disk-source orbits with many spawn steps
sim.advance(orbits=8, spawns=40, orbits_reference='disk_source', verbose=False)
print(f"Total particles: {sim.N}, test particles: {sim.N - sim.N_active}")

## 5. Black-Hole-Centric Plots
The standard SerpensAnalyzer is hard-coded for planetary systems (uses primary radius for plot scales, draws shadow polygons assuming a star-illuminated planet). For black hole systems we plot directly from particle data, in BH-centric coordinates with distances in units of Schwarzschild radii.

In [ ]:
import matplotlib.pyplot as plt

# Extract particle positions/velocities in BH-centric frame
bh = sim.particles['bh']
ds = sim.particles['disk_source']
r_s = sim._r_schwarzschild
r_isco = sim._r_isco

xs = np.array([sim.particles[i].x - bh.x for i in range(sim.N_active, sim.N)])
ys = np.array([sim.particles[i].y - bh.y for i in range(sim.N_active, sim.N)])
zs = np.array([sim.particles[i].z - bh.z for i in range(sim.N_active, sim.N)])
vxs = np.array([sim.particles[i].vx for i in range(sim.N_active, sim.N)])
vys = np.array([sim.particles[i].vy for i in range(sim.N_active, sim.N)])
vzs = np.array([sim.particles[i].vz for i in range(sim.N_active, sim.N)])

rs_dist = np.sqrt(xs**2 + ys**2 + zs**2)
speeds = np.sqrt(vxs**2 + vys**2 + vzs**2)

ds_offset = np.array([ds.x - bh.x, ds.y - bh.y, ds.z - bh.z])
ds_orb_radius = np.linalg.norm(ds_offset)

print(f"Particles: {len(xs)}")
print(f"BH-centric distance range: {rs_dist.min()/r_s:.1f} to {rs_dist.max()/r_s:.1f} r_s")
print(f"Disk source orbit radius: {ds_orb_radius/r_s:.1f} r_s")

### Planar view: full system + zoom near BH

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor='black')
for ax in axes:
    ax.set_facecolor('black')
    ax.tick_params(colors='white')
    for s in ax.spines.values():
        s.set_color('white')

# Full system view
ax = axes[0]
sc = ax.scatter(xs/r_s, ys/r_s, c=speeds/1e6, s=2, alpha=0.7, cmap='plasma')
ax.add_patch(plt.Circle((0, 0), 1, color='white', zorder=10))
ax.add_patch(plt.Circle((0, 0), r_isco/r_s, fill=False, color='red', linestyle='--', label='ISCO'))
ax.plot(ds_offset[0]/r_s, ds_offset[1]/r_s, 'g^', markersize=12, label='disk source')
ax.add_patch(plt.Circle((0, 0), ds_orb_radius/r_s, fill=False, color='green',
                         linestyle=':', alpha=0.4, label='source orbit'))
extent = 1.4 * ds_orb_radius/r_s
ax.set_xlim(-extent, extent)
ax.set_ylim(-extent, extent)
ax.set_aspect('equal')
ax.set_xlabel(r'$x / r_s$', color='white', fontsize=14)
ax.set_ylabel(r'$y / r_s$', color='white', fontsize=14)
ax.set_title(f'Full system ({len(xs)} particles)', color='white', fontsize=14)
cb = plt.colorbar(sc, ax=ax)
cb.set_label('speed [10⁶ m/s]', color='white')
cb.ax.tick_params(colors='white')
ax.legend(facecolor='black', edgecolor='white', labelcolor='white', loc='upper right')

# Zoom near BH (event horizon, ISCO, photon sphere)
ax = axes[1]
ax.scatter(xs/r_s, ys/r_s, c=speeds/1e6, s=10, alpha=0.7, cmap='plasma')
ax.add_patch(plt.Circle((0, 0), 1, color='white', zorder=10, label='Event horizon (1 r_s)'))
ax.add_patch(plt.Circle((0, 0), 1.5, fill=False, color='cyan', linestyle=':', label='Photon sphere (1.5 r_s)'))
ax.add_patch(plt.Circle((0, 0), r_isco/r_s, fill=False, color='red', linestyle='--', label='ISCO (3 r_s)'))
ax.set_xlim(-15, 15)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.set_xlabel(r'$x / r_s$', color='white', fontsize=14)
ax.set_ylabel(r'$y / r_s$', color='white', fontsize=14)
ax.set_title('Zoom near BH (any infallen particles?)', color='white', fontsize=14)
ax.legend(facecolor='black', edgecolor='white', labelcolor='white', loc='upper right')

plt.tight_layout()
plt.show()

### Density maps: planar (x-y) and line of sight (y-z)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor='black')

extent_rs = 1.4 * ds_orb_radius / r_s
edges = np.linspace(-extent_rs, extent_rs, 100)

# Planar x-y
ax = axes[0]
ax.set_facecolor('black')
H, xe, ye = np.histogram2d(xs/r_s, ys/r_s, bins=[edges, edges])
im = ax.imshow(np.log10(H.T + 0.5), origin='lower',
               extent=[xe[0], xe[-1], ye[0], ye[-1]], cmap='afmhot', aspect='equal')
ax.add_patch(plt.Circle((0, 0), 1, color='white', zorder=10))
ax.add_patch(plt.Circle((0, 0), r_isco/r_s, fill=False, color='red', linestyle='--'))
ax.plot(ds_offset[0]/r_s, ds_offset[1]/r_s, 'g^', markersize=10)
ax.set_xlabel(r'$x / r_s$', color='white', fontsize=14)
ax.set_ylabel(r'$y / r_s$', color='white', fontsize=14)
ax.set_title('Planar density (log₁₀ count)', color='white', fontsize=14)
ax.tick_params(colors='white')
cb = plt.colorbar(im, ax=ax)
cb.ax.tick_params(colors='white')

# Line of sight y-z
ax = axes[1]
ax.set_facecolor('black')
H, ye, ze = np.histogram2d(ys/r_s, zs/r_s, bins=[edges, edges])
im = ax.imshow(np.log10(H.T + 0.5), origin='lower',
               extent=[ye[0], ye[-1], ze[0], ze[-1]], cmap='afmhot', aspect='equal')
ax.add_patch(plt.Circle((0, 0), 1, color='white', zorder=10))
ax.add_patch(plt.Circle((0, 0), r_isco/r_s, fill=False, color='red', linestyle='--'))
ax.set_xlabel(r'$y / r_s$', color='white', fontsize=14)
ax.set_ylabel(r'$z / r_s$', color='white', fontsize=14)
ax.set_title('Line of sight (y-z)', color='white', fontsize=14)
ax.tick_params(colors='white')
cb = plt.colorbar(im, ax=ax)
cb.ax.tick_params(colors='white')

plt.tight_layout()
plt.show()

### Radial profile and Keplerian / GR signatures
The Keplerian relation `v ∝ r⁻¹/²` should hold; deviations near the BH indicate GR effects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Radial distribution histogram
ax = axes[0]
ax.hist(rs_dist/r_s, bins=60, color='steelblue', edgecolor='black')
ax.axvline(1, color='black', linestyle='--', label='Event horizon')
ax.axvline(r_isco/r_s, color='red', linestyle='--', label='ISCO')
ax.axvline(ds_orb_radius/r_s, color='green', linestyle=':', label='disk source orbit')
ax.set_xlabel(r'distance from BH $[r_s]$')
ax.set_ylabel('particle count')
ax.set_title('Radial particle distribution')
ax.legend()

# Speed vs radial distance with Keplerian comparison
ax = axes[1]
ax.scatter(rs_dist/r_s, speeds/1e6, s=3, alpha=0.4, label='particles')
r_fine = np.linspace(rs_dist.min(), rs_dist.max(), 200)
v_kepler = np.sqrt(G * bh.m / r_fine)
ax.plot(r_fine/r_s, v_kepler/1e6, 'r--', linewidth=2, label='Keplerian $v=\\sqrt{GM/r}$')
ax.axvline(r_isco/r_s, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel(r'distance from BH $[r_s]$')
ax.set_ylabel('speed [10⁶ m/s]')
ax.set_title('Speed vs radial distance')
ax.legend()

plt.tight_layout()
plt.show()

### Interactive 3D plot

In [ ]:
import plotly.graph_objects as go

# Event horizon sphere (parametric)
phi, theta = np.mgrid[0:2*np.pi:30j, 0:np.pi:15j]
xh = np.cos(phi) * np.sin(theta)
yh = np.sin(phi) * np.sin(theta)
zh = np.cos(theta)

# ISCO sphere wireframe
xi = (r_isco/r_s) * xh
yi = (r_isco/r_s) * yh
zi = (r_isco/r_s) * zh

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=xs/r_s, y=ys/r_s, z=zs/r_s,
    mode='markers',
    marker=dict(size=2, color=speeds/1e6, colorscale='Plasma', colorbar=dict(title='speed [10⁶ m/s]')),
    name='particles'
))
fig.add_trace(go.Surface(x=xh, y=yh, z=zh, showscale=False, opacity=0.9,
                         colorscale=[[0, 'black'], [1, 'black']], name='event horizon'))
fig.add_trace(go.Surface(x=xi, y=yi, z=zi, showscale=False, opacity=0.15,
                         colorscale=[[0, 'red'], [1, 'red']], name='ISCO'))
fig.add_trace(go.Scatter3d(
    x=[ds_offset[0]/r_s], y=[ds_offset[1]/r_s], z=[ds_offset[2]/r_s],
    mode='markers', marker=dict(size=8, color='green', symbol='diamond'),
    name='disk source'
))
fig.update_layout(
    template='plotly_dark',
    title='SERPENS Black Hole — 3D particle distribution (units of r_s)',
    scene=dict(
        xaxis_title='x / r_s',
        yaxis_title='y / r_s',
        zaxis_title='z / r_s',
        aspectmode='data'
    ),
    height=700
)
fig.show()